### Overview

This notebook runs a trained ASAHI model on low-depth ENCODE libraries and scores the result against the same libraries at full depth.

`checkpoints/asahi_chr21.pth` is the chr21 model behind the paper's ENCODE results. No ENCODE library was used to train it.

#### How inference works
Row 0 of each slice holds the library being predicted, at its real low depth, and only its predictions are kept. The other 7 rows are support libraries drawn at random with replacement, and each patch is predicted 5 times with different draws, then averaged. Nothing is masked: the model sees every observed value and predicts every CpG, denoising those it saw and imputing the rest.

A GPU is strongly recommended.

#### Outputs
`results/encode_pt_frac0.1_seed11_chr21.csv`: one row per CpG (`start`) and one column per library, with betas in [0, 1].

In [1]:
%cd -q ..

In [2]:
import torch
from scripts.inference import run_inference
from scripts.evaluate import evaluate

### Load a pretrained model
A checkpoint holds the model's hyperparameters and weights. `run_inference` below loads it the same way, then attaches the chromosome's encoded genome, which is not stored in the checkpoint.

In [3]:
from asahi.model import load_checkpoint

model = load_checkpoint("checkpoints/asahi_chr21.pth", map_location="cpu")
print(f"{type(model).__name__}: {sum(p.numel() for p in model.parameters()):,} parameters")
print(model.hparams)

ASAHI: 4,395,787 parameters
namespace(cpg_embed_size=32, transf_hsz=64, n_transformers=4, n_heads=8, head_dim=8, window=21, transf_do=0.2, act='relu', dna_embed_size=32, dna_window=1001, dna_conv_do=0.2, dna_chunks=16, depth_embed_size=8, depth_cap=32, aux_local_bp=500, aux_local_weight=0.25, aux_cross_weight=0.25, aux_warmup_epochs=10, lr=0.0005, lr_decay_factor=0.998719, warmup_steps=1000)


### Predict all 12 libraries at 3x
The fractions 0.1, 0.2, 0.4 and 0.6 of the reads are labelled 3x, 6x, 12x and 18x in the paper. To try another, change `input_dataset`.

In [4]:
input_dataset = "encode_pt_frac0.1_seed11"
device = "cuda:0" if torch.cuda.is_available() else "cpu"
preds = run_inference(checkpoint="checkpoints/asahi_chr21.pth",
                      chrom="chr21",
                      anchor_dataset=input_dataset,
                      support_dataset="encode_pt",
                      out=f"results/{input_dataset}_chr21.csv",
                      device=device)

[INFO] 06-Nov-26 20:37:50 reading data/encode_pt_frac0.1_seed11/chrom_centric/true/FractionalMethylation/chr21.fm.gz


[INFO] 06-Nov-26 20:37:51 reading data/encode_pt_frac0.1_seed11/chrom_centric/true/ReadDepth/chr21.cov.gz


[INFO] 06-Nov-26 20:37:51 chr21: 462299 CpGs x 12 libraries -> 452 patches of 1024 CpGs


[INFO] 06-Nov-26 20:37:51 reading data/encode_pt/chrom_centric/true/FractionalMethylation/chr21.fm.gz


[INFO] 06-Nov-26 20:37:52 reading data/encode_pt/chrom_centric/true/ReadDepth/chr21.cov.gz


[INFO] 06-Nov-26 20:37:52 chr21: 462299 CpGs x 12 libraries -> 452 patches of 1024 CpGs


... (9 outputs left out)

[INFO] 06-Nov-26 20:43:21 batch 1352/1695


[INFO] 06-Nov-26 20:44:01 batch 1521/1695


[INFO] 06-Nov-26 20:44:41 batch 1690/1695


[INFO] 06-Nov-26 20:44:49 wrote results/encode_pt_frac0.1_seed11_chr21.csv (462299 x 12)


### Score against full depth
Cells count only where the full-depth read depth is at least 10. *Denoised* cells were observed in the low-depth input and *imputed* cells were not. `Obs` is the low-depth input left as is, so it has no imputed cells. Each value is the mean over libraries of the per-library MSE.

In [5]:
per_library, summary = evaluate(preds, "chr21", input_dataset, "encode_pt")
summary

[INFO] 06-Nov-26 20:44:49 reading data/encode_pt/chrom_centric/true/FractionalMethylation/chr21.fm.gz


[INFO] 06-Nov-26 20:44:50 reading data/encode_pt/chrom_centric/true/ReadDepth/chr21.cov.gz


[INFO] 06-Nov-26 20:44:50 chr21: 462299 CpGs x 12 libraries -> 452 patches of 1024 CpGs


[INFO] 06-Nov-26 20:44:50 reading data/encode_pt_frac0.1_seed11/chrom_centric/true/FractionalMethylation/chr21.fm.gz


[INFO] 06-Nov-26 20:44:51 reading data/encode_pt_frac0.1_seed11/chrom_centric/true/ReadDepth/chr21.cov.gz


[INFO] 06-Nov-26 20:44:51 chr21: 462299 CpGs x 12 libraries -> 452 patches of 1024 CpGs


,subset,method,libraries,mean_cells,mean_mse
0,denoised,ASAHI,12,288893.166667,0.019187
1,denoised,Obs,12,288893.166667,0.044070
2,imputed,ASAHI,12,27407.416667,0.023033


In [6]:
per_library[per_library.subset == "denoised"].pivot(index="library", columns="method", values="mse").round(4)

method,ASAHI,Obs
library,,
ENCFF032LIY,0.0175,0.0425
ENCFF055UXZ,0.0201,0.0474
ENCFF116KBT,0.0203,0.0499
ENCFF164ZJM,0.0266,0.0600
ENCFF249ESJ,0.0246,0.0612
ENCFF337XLQ,0.0233,0.0570
ENCFF357SKM,0.0201,0.0438
ENCFF435IGQ,0.0113,0.0216
ENCFF473TTY,0.0156,0.0299
